In [1]:
import numpy as np
from scipy.sparse.linalg import gmres
from sympy import init_printing,Matrix#for printing nice matrices

In [2]:
A = np.array([[30, 20, 1e-6], [1, -1, 1], [333, 5, 1]])
b = np.array([4560, 10, 4])
x, Ec = gmres(A, b, atol=1e-6)

In [3]:
x

array([ -4.25386974, 234.38079218, 248.63466191])

Define the problem 

$$
A_{ij} x_j = b_i
$$

$A$ ($N \times N$ matrix) and $b$ (vector of dimention $N$) are known, $x$ (vector of dimention $N$) is the solution I am looking for.



In [2]:
#This defines the product between matrix and vector. GMRES only need vectors A.v, so this will be the only thing I use.
def matvec(A,v):
    N=len(A)
    c=np.zeros(N)
    for i in range(N):
        for j in range(N):
            c[i]+=A[i][j]*v[j]
    return c    

#I will take iterative steps to find a small enough residual 
def residual(A,b,test_solution):
    return b - matvec(A,test_solution)

#I will need also the norm of vectors
def norm(v):
    n=0
    for i in range(len(v)):
        n+=v[i]**2.
    return n**0.5

def vector_scale(v,a):
    N=len(v)
    v_div_a=np.zeros(N)
    for i in range(N):
        v_div_a[i]=v[i]/a
    return v_div_a

def dot(v,u):
    N=len(v)
    prod=0
    for i in range(N):
        prod+=v[i]*u[i]
    return prod      

In [9]:
#setup the problem
N=5
A=np.random.uniform(-5,5,[N,N])
b=np.random.uniform(-5,5,N)



# GMRES builds a vector space of some dimension m (user choice).
# Then, it looks for a vector in this space that approximates the solution of A x =b.
subspace_dimension=N #choosing N, basically we are guaranteed to find a solution because we build basis for the vector space of b.

#this will be the basis of the vector space I will be creating
v=np.zeros([subspace_dimension,N])
# This matrix ( dimension: (m+1) x m ) is useful in things we will see later.
h=np.zeros([subspace_dimension+1,subspace_dimension])

In [10]:
# To start GMRES, we take an initial guess. 
# This allows us to build the first basis vector

#choose an initial guess
x0=np.zeros_like(b)

#compute the residual
res=residual(A,b,x0)
beta=norm(res)

#first basis vector
v[0] = vector_scale(res,beta) 

In [11]:
# From the first basis vector, we can start constructing the next.

# This builds the basis. It is essentially Gram-Schmidt.
# the only difference is that the initial vectors
# are built from A.v_j 
for j in range(1,subspace_dimension):
    w=matvec(A,v[j-1])
    
    for i in range(j):
        w -= v[i]*dot(v[i],w)

    v[j] = vector_scale(w,norm(w)) 

# this is the main loop. GMRES will try to find approximation 
# for each new basis vector I construct. So, I use this as the 
# main loop. inside thism I will run the other things. 

In [12]:
#Another thing I need to understand is solving triangular systems. 
# It is easy because a triangular matrix is a matrix with:
# R_{ij}=0 for i>j

#Example:
R=np.zeros([N,N])
for i in range(N):
    for j in range(i+1):
        R[j][i]=np.random.rand()
R

array([[0.38716791, 0.39552621, 0.34409035, 0.11559971, 0.8616426 ],
       [0.        , 0.35924831, 0.32293581, 0.408615  , 0.14554811],
       [0.        , 0.        , 0.65972722, 0.11487716, 0.97735104],
       [0.        , 0.        , 0.        , 0.12880687, 0.41903859],
       [0.        , 0.        , 0.        , 0.        , 0.39641437]])

In [13]:
# If I want to solve Ry=d, I can start with the Nth element that has a solution 
# y_{N}=d_N/R_{NN}
# Since, I have y_N,  I can solve is 
# R_{N-1,N-1}y_{N-1}+R_{N-1,N}y_{N}=d_{N-1} =>  y_{N-1} = (d_{N-1} - R_{N-1,N}y_{N})/R_{N-1,N-1}
# And I continue. This is called backsubstitution.
# The general formula is

# y_{N}=d_N/R_{NN}
# y_{N-i} = (d_{N-i} - \sum_{j>N-i}  R_{N-1,N}y_{N})/R_{N-i,N-i}

In [3]:
# Let's test it
def triangular_solve(R,d):
    N=len(d)
    y=np.zeros(N)
    for i in range(1,N+1):
        s=0
        for j in range(N-i+1,N):
            s+=R[N-i][j]*y[j]
        y[N-i]=(d[N-i]-s)/R[N-i][N-i]

    return y

In [15]:
R=np.zeros([N,N])
d=np.zeros(N)
for i in range(N):
    d[i]=np.random.uniform(-1,1)
    for j in range(i+1):
        R[j][i]=np.random.uniform(-1,1)

for i in range(N):
    R[i][i]+=10

x=np.array(triangular_solve(R,d))


norm(R@x-d)/norm(d)

np.float64(0.0)

# Subdtle point: 
The number of vectors I build is 1 more that the number of the subspace! 
So, if the dimension of the subspace is equal to the demension of the problem, the final 
would-be basis vector should have zero norm because I have exhausted the number of 
linearly independent vectors.

Apparently, this extra vector makes things easier for the later step of least squares.

The problem, not has to be reformulated in order to take into account this extra vector

In [ ]:
#setup the problem
N=8
A=np.random.uniform(-5,5,[N,N])
b=np.random.uniform(-5,5,N)



# GMRES builds a vector space of some dimension m (user choice).
# Then, it looks for a vector in this space that approximates the solution of A x =b.
subspace_dimension=4 #choosing N, basically we are guaranteed to find a solution because we build basis for the vector space of b.

#this will be the basis of the vector space I will be creating
v=np.zeros([subspace_dimension+1,N])



# triangular problems are nice because they can be solved exaclty.
# If you think about it, there is an *almost* triangular matrix that is related
# to the Arnoldi basis-building iteration. So, let's break it down, by
# saving the different norms to a matrix we call h.
#Basically, h allow us to decompose A into components of v.

h=np.zeros([subspace_dimension+1,subspace_dimension])

# To start GMRES, we take an initial guess. 
# This allows us to build the first basis vector

#choose an initial guess
x0=np.zeros_like(b)+1

#compute the residual
res=residual(A,b,x0)
beta=norm(res)

#first basis vector
v[0] = vector_scale(res,beta) 

In [19]:
for j in range(subspace_dimension):
    #new vector (this will be the next basis vector after orthonormalization)
    w = matvec(A, v[j])

    #orthogonalize v_0, v_1, ... v_j
    for i in range(j + 1):
        h[i][j] = dot(v[i], w)
        w -= h[i][j] * v[i]

    h[j + 1][j] = norm(w)

    #if the norm vanishes, we have exhausted the linearly independent
    #vectors we can have from the guess x0. Maybe we need to restart
    #with a different guess (if solution cannot be found).
    if h[j + 1][j] < 1e-14:
        break

    # normalize next basis vector
    v[j + 1] = vector_scale(w, h[j + 1][j])  # if vector_scale = divide by scalar

In [20]:
# check orthonormalization (only diagonal terms should be non-zero, and equal to 1)
for i in range(subspace_dimension+1):
    for j in range(subspace_dimension+1):
        prod=dot(v[i],v[j])
        if np.abs(prod)>1e-8:
            print(i,j,prod)

0 0 1.0
1 1 1.0
2 2 1.0
3 3 1.0000000000000002
4 4 1.0


In [48]:
# this decomposition relates A,v, and h via
# Av_{j}=v_{j+1} h_{j+1,j} + \sum_{i<=j} v_{i} h_{i,j}

#everything should be close to 0
for j in range(subspace_dimension):
    s=0
    for i in range(j+1):
        s+= v[i] * h[i][j]
    print(norm(A@v[j] - v[j+1] * h[j + 1][j] - s))

1.0692427990109977e-15
1.4111738243378813e-15
2.094764613337708e-15
1.2755491433176288e-15


# Let's start exploring the idea of GMRES

## Notation for the objects

- $N$: number of equations
- $m$: dimension of subspace, $m \leq N$.

The problem is defined as $A_{ij} x_j = b_i$.

The Arnoldi loop builds $m+1$ basis vectors, $v^{(1)}$, $v^{(2)}$ ,..., $v^{(m+1)}$.

During Arnoldi, we also fill the "Hessenberg" matrix $h_{rj}$ with $r=1,2,...,m+1$ and $j=1,2,...,m$. This matrix holds the projection of $A$ the basis vectors $v$ because if comes from the "Gram-Schmidt"-like Arnoldi loop:

---

Start with  $v^{(1)}_i= (b_i - A_{ij} x^{0}_j)/\beta$ and $\beta = |(b_i - A_{ij} x^{0}_j)|$.

For each j=1,2,...,m, we construct the next basis vectors ($v^{(j+1)}$) starting from $w_{k}^{(j+1)}$:

- $w_{k}^{(j+1)} \leftarrow  \sum_{l=1}^{N} A_{kl} v^{(j)}_{l}$ (for all k=1,2,...,N)

Then, we subtract the projections of all previous basis vectors ($v^{(1)}$, $v^{(2)}$,..., $v^{(j)}$) from $w_{k}^{(j+1)}$.

- For i=1,2,...,j
- - $h_{ij} \leftarrow \sum_{k=1}^{N}  v^{(i)}_k w_{k}^{(j+1)}$

- - $w_{k}^{(j+1)} \leftarrow  w_{k}^{(j+1)} - h_{ij} \, v^{(i)}_k $ (k=1,2,...,N)

<p style="text-align: right;">
This loop basically constructs  $ T_k = w_{k}^{(j+1)} - \sum_{i=1}^{j} v^{(i)}_k \sum_{l=1}^{N}  v^{(i)}_l w_{l}^{(j+1)} $, which obeys (for $p<j+1$)
</p>
<p style="text-align: right;">
$
 \sum_{k=1}^{N} v^{(p)}_k T_k   = \sum_{k=1}^{N} v^{(p)}_k w_{k}^{(j+1)} - \sum_{k=1}^{N} v^{(p)}_k \sum_{i=1}^{j} v^{(i)}_k \sum_{l=1}^{N}  v^{(i)}_l w_{l}^{(j+1)}= 
$
</p>

<p style="text-align: right;">
$
\sum_{k=1}^{N} v^{(p)}_k w_{k}^{(j+1)} - \sum_{i=1}^{j} \delta_{ip} \sum_{l=1}^{N}  v^{(i)}_l w_{l}^{(j+1)} =
$ 
</p>
<p style="text-align: right;">
$
\sum_{k=1}^{N} v^{(p)}_k w_{k}^{(j+1)} -  \sum_{l=1}^{N}  v^{(p)}_l w_{l}^{(j+1)} = 0
$
</p>

Then, we normalize  $w_{k}^{(j+1)}$ using

- $h_{j+1 \, j} \leftarrow |\vec{w}^{(j+1)}|$ 
- $v_{k}^{(j+1)} \leftarrow w_{k}^{(j+1)}/h_{j+1 \, j}$ (k=1,2,...,N)

---

In this loop, we basically built:

for j=1,2,...,m:
- $h_{1j}$, $h_{2j}$,..., $h_{jj}$, $h_{j+1\,j}$.
- $v^{(j+1)}$


As mentioned before the start of the loop, this allows us to decompose

$$
\sum_{l=1}^{N} A_{kl} v^{(j)}_{l}=v_k^{(j+1)} h_{j+1\,j} + \sum_{i=1}^j v_k^{(i)} h_{ij} = \sum_{i=1}^{j+1} v_k^{(i)} h_{ij}
$$

Not that this is true during the iteration because each new basis vector is orthogonalized with the previous ones.

Why do all that?

The idea behind GMRES is to build an iteration of the form

$$
x_k = x_k^{(0)} + \sum_{r=1}^{m} v^{(r)}_{k} \, y_{r},
$$

where $y$ is a vector in the v-basis; i.e. it is a vector of the same dimension as the subspace we are building. 

This helps because it reduces the space in which we are exploring while we construc an approximate solution. Basically, we adjust $y$ to minimize the norm of the resudual vector $r$,

$$
r_{k}(y)=b_{k}−\sum_{l=1}^N A_{kl} x_l = b_{k}−\sum_{l=1}^N A_{kl}
( x_l^{(0)} + \sum_{r=1}^{m} v^{(r)}_{l} \, y_{r} )
$$

Using $b_{k}−\sum_{l=1}^N A_{kl} x_l^{(0)}  = \beta \, v^{(1)}_k = \beta \sum_{j=1}^{m+1} v^{(j)}_k \delta_{j1}$, we can rewrite $r$ as


$$
r_{k}(y)=\beta \sum_{j=1}^{m+1} v^{(j)}_k \delta_{j1} − \sum_{l=1}^N A_{kl} \sum_{r=1}^{m} v^{(r)}_{l} \, y_{r} =
\beta \sum_{j=1}^{m+1} v^{(j)}_k \delta_{j1} −  \sum_{r=1}^{m}\sum_{j=1}^{r+1} v_k^{(j)} h_{jr} \, y_{r}
$$

Since $h_{jr}=0$ for $j>r+1$, I can run the sum over $j$ up to $m+1$ (r goes up to $m$, so $j$ should go up to max($r$)$+1=m+1$). That is

$$
r_{k}(y)=
\beta \sum_{j=1}^{m+1} v^{(j)}_k \delta_{j1} −  \sum_{r=1}^{m}\sum_{j=1}^{m+1} v_k^{(j)} h_{jr} \, y_{r}=
 \sum_{j=1}^{m+1} v^{(j)}_k (\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})
$$

So, minimizing $|r(y)|$ means minimizing $|\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r}|$!

That is, we are looking for the minimum of

$$
\sum_{j=1}^{m+1} (\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})^2
$$

or 

$$
\dfrac{d}{d y_p}\sum_{j=1}^{m+1} (\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})^2=0
$$


That is

\begin{eqnarray}
&\sum_{j=1}^{m+1} \dfrac{d}{d y_p}(\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})^2=\\
&\sum_{j=1}^{m+1} 2(\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})\dfrac{d}{d y_p}(\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})= \\
&−  \sum_{j=1}^{m+1} 2(\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})  \sum_{r=1}^{m} h_{jr} \, \delta_{rq} =\\
&−  \sum_{j=1}^{m+1} 2(\beta \delta_{j1} −  \sum_{r=1}^{m} h_{jr} \, y_{r})   h_{jq}  =\\
&−   2( \beta h_{1q} −  \sum_{j=1}^{m+1}\sum_{r=1}^{m} h_{jq} h_{jr} \, y_{r})     =0
\end{eqnarray}

Or

$$
\beta h_{1q} = \sum_{i=1}^{m+1}\sum_{j=1}^{m} h_{iq} h_{ij} \, y_{j}
$$

Instead of first building the basis, and then solving the system, GMRES, minimizes $r(y)$ for each new basis vector it introduces. That is, at the end of the $k^th$ Arnoldi iteration (once $v^{(k+1)}$ is computed), we solve 


$$
\beta h_{1q} = \sum_{i=1}^{k+1}\sum_{j=1}^{k} h_{iq} h_{ij} \, y_{j} 
$$

for $q=1,2,..,k$ (since $k$ is the current subspace dimension).

The system can also be written in a matrix friendly form:

$$
\beta h_{1q} = \sum_{j=1}^{k} M_{qj} \, y_{j}
$$

with $M_{qj} = \sum_{i=1}^{k+1} h_{iq} h_{ij}$, a $k \times k$ symmetric matrix.

---

Notice the subdlety of $j=1,2,...,k+1$. Although the subspace has dimension $k$, we also take into account the "remainder" of $A v^{(k)}$ on the subspace. That is, value of k at which $h_{k+1 \, k}\approx 0$ tells us how close we are in covering the subspace defined through $A$, $b$, and the initial guess solution $x^{(0)}$. This is why if $h_{k+1 \, k} \approx 0$, we stop (or restart) Arnoldi. 

In principle, we can do the following:

$c_q=\beta h_{1q}$

$c_q=\sum_{j=1}^{k} M_{qj} \, y_{j}$

For p=1,2,...,k-1

- For r=p+1,p+2,...,k
  
- - $\lambda_{rp} = \dfrac{M_{rp}}{M_{pp}}$

- - For j=p,p+1,...,k
- - - $M_{rj} \leftarrow M_{rj} - \lambda_{rp} M_{pj}$ 
- - - $c_r \leftarrow c_r - \lambda_{rp} c_p $

At the end, the transformed $M_{ij}$ is upper triangular, and the system can be solved using 

$y_k=c_k/M_{kk}$

Since we have $c_k$,  we can solve  

$$ M_{k-1,k-1}y_{k-1}+M_{k-1,k}y_{k}=c_{k-1} =>  y_{k-1} = (c_{k-1} - M_{k-1,k}y_{k})/M_{k-1,k-1} $$

And continue. This is called backsubstitution, with general formula

$y_k=c_k/M_{kk}$

$$y_{k-i} = (c_{k-i} - \sum_{j=k-i+1}^{k}  M_{k-i,j}y_{j})/M_{k-i,k-i}$$

The  backsubstitution is generally unstable (e.g. $M_{ii}$ could have small entries that causes errors to builr up). So, we prefere a different way, using "Givens rotations". Instead of solving $\dfrac{d r(y)}{dy}=0$, we start with the original question

$$
{\rm min}_y \left\{ \sum_{i=1}^{k+1} \left(\beta \delta_{i1} −  \sum_{j=1}^{k} h_{ij} \, y_{j} \right)^2 \right\}
$$


The way we do it is simple. Start with  the vector 
$$T_i = g_{i} −  \sum_{j=1}^{k} h_{ij} \, y_{j}$$   
with $g_i=\beta \delta_{i1}$.

Perform orthogonal transformation $\hat O$ (sum over i is implied)

$$O_{li}T_{i} = O_{li}g_{i} −  \sum_{j=1}^{k} (O_{li}h_{ij}) \, y_{j} $$

The goal is to make $\tilde h = \hat O \, h$ upper triangular. By construction, $h_{ik}=0$ for $i>k+1$. So, we only need to rotate for $i=k+1$ and make $\tilde h_{k+1\,k}=0$. 

To do this during the Arnoldi loop, we note that we can decompise $\hat O = \hat O^{(k)} \, \hat O^{(k-1)} \,... \hat O^{(1)}$, with each $\hat O^{(p)}$ a rotation of the plane $p$ and $p+1$. This means that  

$$
\hat O^{(p)}_{ij}=\left( \begin{matrix} 
                        c_p & s_p \\ 
                        -s_p & c_p 
                    \end{matrix}\right), \quad {\rm for}\, (i,j)=(p,p+1)
$$

and for all other values of $i$ and $j$,
$$
O^{(p)}_{ij} = \delta_{ij} 
$$

Then,  during the $k^{th}$ step of Arnoldi loop, we compute

\begin{eqnarray}
&\tilde h_{k\,k} = O^{(k)}_{kj}h_{jk}= c_k \, h_{kk} + s_k \, h_{k+1\,k} = r
\\
&\tilde h_{k+1\,k} = O^{(k)}_{k+1\,j}h_{jk}= -s_k \, h_{kk} + c_k \, h_{k+1\,k} = 0
\end{eqnarray}

This means that $c_k=h_{kk}/r$, $s_k=h_{k+1\,k}/r$, and $r^2=h^2_{k\,k}+h^2_{k+1\,k}$ (so that $c_k^2+s_k^2=1$).

## A subdle point.

At the $k^{th}$ step of the Arnoldi loop, you create $h_{k,k}$, $h_{k+1,k}$, and also all $h_{i,k}$ with $i=1,2,...,k-1$. This means that you need to apply $\hat O^{(k-1)}$ to those, to keep the transportation fully orthodonal. That is, every step, after we create $h_{1\,k}$, $h_{2\,k}$,..., $h_{k-1\,k}$, $h_{k\,k}$, $h_{k+1\,k}$, we apply the rotation 
$$
O_{li}h_{ik}=O^{(k)}_{l\,i_k} O^{(k-1)}_{i_{k}\,i_{k-1}}...O^{(2)}_{i_3\,i_2} O^{(1)}_{i_2\,i_1} h_{i_{1}\, k} 
$$

this gives us the "algorithm":

At step $k$:
- for i in 1,2,...,k-1

- - compute $\tilde h_{lk} = O^{(i)}_{l\,j} h_{j\, k}$. Here the transformation only applies for $l=i,i+1$  (all others indices result in multiplication with $\delta_{li}$). So, I compute

\begin{eqnarray}
&\tilde h_{i\,k} = O^{(i)}_{ij}h_{jk}= c_i \, h_{ik} + s_i \, h_{i+1\,k}  
\\
&\tilde h_{i+1\,k} = O^{(i)}_{i+1\,j}h_{jk}= -s_i \, h_{ik} + c_i \, h_{i+1\,k}  
\end{eqnarray}

Once this loop exits, we have transformed all $h_{ik}$ up to $h_{k-1\,k}$. For the $k^{th}$ component, we impose

\begin{eqnarray}
&\tilde h_{k\,k} = O^{(k)}_{kj}h_{jk}= c_k \, h_{kk} + s_k \, h_{k+1\,k} = r
\\
&\tilde h_{k+1\,k} = O^{(k)}_{k+1\,j}h_{jk}= -s_k \, h_{kk} + c_k \, h_{k+1\,k} = 0
\end{eqnarray}

This gives us the values of $O^{(k)}$, which will be used in the next iteration duing the "$i$" loop.

To keep the transformation of $T_i$ consistent, we also transform (at every step $k$) $g_k$ with

$$
\tilde g_k = O^{(k)}_{ki}g_{i},\qquad
\tilde g_{k+1} = O^{(k)}_{k+1i}g_{i}
$$


We can finally find y that minimizes (using $h_{ij}=0$ for $j<i$)

$$
 \sum_{i=1}^{k+1} \left(g_i −  \sum_{j=1}^{k} h_{ij} \, y_{j} \right)^2 =
 \sum_{i=1}^{k+1} \left(g_i −  \sum_{j=i}^{k} h_{ij} \, y_{j} \right)^2 =
 \sum_{i=1}^{k} \left(g_i −  \sum_{j=i}^{k} h_{ij} \, y_{j} \right)^2 + g_{k+1}^2
$$

The last term is there because for $i=k+1$, $\sum_{j=i}^{k} h_{ij} \, y_{j}=0$.

Since each term is positive, minimization wrt to y means minimization of $\sum_{i=1}^{k} \left(g_i −  \sum_{j=i}^{k} h_{ij} \, y_{j} \right)^2$. There are enough free components to make this $0$, so we just solve

$$
\sum_{j=i}^{k} h_{ij} \, y_{j}=g_i
$$

Since $h$ is triangular, we can solve it using "backsubstitution"

In [9]:
# setup the problem again and rewrite the Arnoldi loop
#setup the problem
N=15
A=np.random.uniform(0,5,[N,N])
A=A+A.T
# A=np.eye(N,N)+7e-2*np.random.uniform(-5,5,[N,N])

rhs=np.random.uniform(-1,1,N)

subspace_dimension=10#choose it to be smaller than N. This is what makes GMRES faster than LU or other algorithms

v=np.zeros([subspace_dimension+1,N])

h=np.zeros([subspace_dimension+1,subspace_dimension])

x=np.zeros_like(rhs)
for itaration in range(100):
    
    v=np.zeros([subspace_dimension+1,N])
    h=np.zeros([subspace_dimension+1,subspace_dimension])

    #compute the residual
    res=residual(A,rhs,x)
    beta=norm(res)
    
    #first basis vector
    v[0] = vector_scale(res,beta) 
    
    e1=np.zeros(subspace_dimension+1)
    e1[0]=1
    g=beta*e1
    c=np.zeros(subspace_dimension+1)
    s=np.zeros(subspace_dimension+1)
    
    # for testing I keep the original h and g.
    h_original=np.zeros([subspace_dimension+1,subspace_dimension])
    g_original=g[:]
    
    for k in range(subspace_dimension):
        #new vector (this will be the next basis vector after orthonormalization)
        w = matvec(A, v[k])
    
        #orthogonalize v_0, v_1, ... v_j
        for i in range(k + 1):
            
            h[i][k] = dot(v[i], w)
            w -= h[i][k] * v[i]
            #only for testing
            h_original[i][k]=h[i][k]
    
        h[k + 1][k] = norm(w)
        h_original[k+1][k]=h[k+1][k]
    
        #I will use this later to see if I need to break the loop
        test_h=h[k + 1][k]
        
    
        # normalize next basis vector (if h[k + 1][k]=0, just put v[k + 1]=0)
        if test_h<1e-10:
            pass
            #v[k + 1] will remain equal to 0 (from its initialization)
        else:    
            v[k + 1] = vector_scale(w, h[k + 1][k])  # if vector_scale = divide by scalar
    
        #here, I can apply rotations to h[i][k] and h[k + 1][k]
    
        # first, I apply O^{(i)} i=0,1,...,k-1 (range(k) generates this)
        for i in range(k):
            p1=h[i][k]
            p2=h[i+1][k]
            h[i][k]=c[i]*p1+s[i]*p2
            h[i+1][k]=-s[i]*p1+c[i]*p2
            
            
        
        p1=h[k][k]
        p2=h[k+1][k]
        r=np.sqrt(p1**2+p2**2)
        if r<1e-10:
            k_end=k
            break
            
            c[k]=1
            s[k]=0
        else:
            c[k]=p1/r
            s[k]=p2/r
        h[k][k]=r
        h[k+1][k]=0 # I put 0 explicitely because -s[k]*a+c[k]*b can have roundoff errors
    
        
        # i need to apply the transformation to g as well
        g0=c[k]*g[k]+s[k]*g[k+1]
        g1=-s[k]*g[k]+c[k]*g[k+1]
        g[k]=g0
        g[k+1]=g1
    
        k_end=k+1#this will tell me at which k we exit the loop(in case we exit earlier)
        
        #if the norm of h[k + 1][k] (before transformation) vanishes, we have exhausted the linearly independent
        #vectors we can have from the guess x0. Maybe we need to restart
        #with a different guess (if solution cannot be found).
        if test_h < 1e-10:
            break
    
    #something is wrong with this... write it explicitly...
    # Solve the active triangular system only
    y = triangular_solve(h[:k_end, :k_end], g[:k_end])
    # y=np.linalg.lstsq(h,g)[0]#this is the same 
    
    dx = np.zeros_like(x)
    for j in range(k_end):
        dx += y[j] * v[j]

    x += dx
    print(norm(A@x-rhs),norm(rhs),end='\t')
    print(norm(g[:k_end-1]),g[k_end])
    if norm(A@x-rhs)/norm(rhs) < 1e-5:
        break
    # x0=x[:]
norm(A@x-rhs),norm(rhs)

1.1582160760534084 2.3866887259466276	2.0754643369325225 1.158216076053408
0.8396298918969399 2.3866887259466276	0.7958622055908987 0.8396298918969403
0.6122299913446115 2.3866887259466276	0.5331125211124754 0.6122299913446144
0.5356209543449427 2.3866887259466276	0.2371452065859345 0.5356209543449416
0.47041197004100743 2.3866887259466276	0.2508252711157775 0.4704119700410079
0.4139161086779441 2.3866887259466276	0.1819979257857726 0.4139161086779427
0.3642081763260579 2.3866887259466276	0.19307492395459583 0.36420817632605773
0.3205761374777052 2.3866887259466276	0.14007632089503017 0.3205761374777027
0.28217240449578485 2.3866887259466276	0.14721733253914926 0.28217240449578607
0.24846404348865483 2.3866887259466276	0.10956626685979573 0.2484640434886587
0.21878386604576713 2.3866887259466276	0.11272139630320842 0.21878386604576844
0.19271387192430922 2.3866887259466276	0.08887661407453057 0.19271387192430997
0.1697510751233674 2.3866887259466276	0.08823762484958395 0.16975107512336

(np.float64(2.310567828344704e-05), np.float64(2.3866887259466276))

# Check 

I need to check the $k$ and $k_end$ after the givens rotations and especially the     `triangular_solve` and the update of `x`

In [1086]:
for i in range(subspace_dimension):
    # check that h is upper triangular
    a=np.max(np.abs(h[i+1:,i]))
    if a>0:
        print(i,a)

In [1087]:
# Matrix(h),Matrix(h_original)

for any vector $y$ , $\left\{ \sum_{i=1}^{k+1} \left(g_i −  \sum_{j=1}^{k} h_{ij} \, y_{j} \right)^2 \right\}$ should be te same both for the transformed and the original $h$ and $g$.

In [1088]:
for test_loop in range(10000):
    
    T=np.zeros(subspace_dimension+1)
    T_original=np.zeros(subspace_dimension+1)
    y=np.random.uniform(-300,300,subspace_dimension)
    
    for i,_ in enumerate(T):
        s=0
        s_original=0
        for j in range(subspace_dimension):
            s+=h[i][j]*y[j]
            s_original+=h_original[i][j]*y[j]
        T[i]=g[i]-s
        T_original[i]=g_original[i]-s_original
    
    if np.abs(norm(T)/norm(T_original)-1)>1e-2:
        print(y, np.abs(norm(T)/norm(T_original)-1)*100)